# **Introduction to PyTorch**

As you already know, python has libraries for nearly all tasks. For machine learning, the two most popular ones are [PyTorch](https://pytorch.org/) and [TensorFlow](https://www.tensorflow.org/). Even though TensorFlow may be a little easier to start with, PyTorch offers more flexibility, which is why we like to work with it.

We will show you how to prepare data for neural network training and how to build a simple network. If you want more, we also have a deep dive introduction into `Tensors` and `Modules` [in this notebook](https://github.com/Center-for-Health-Data-Science/IntroToML/blob/main/Day2/intro_to_pytorch_basics_in_detail.ipynb), which are the base for all the helpful neural network building blocks in PyTorch.

We hope you will have fun and find this material useful.

<img src="../figures/HeaDS_logo_large_withTitle.png" width="200">

(This notebook was created by Viktoria Schuster)

In [18]:
# prevent warnings
import warnings
warnings.filterwarnings("ignore")

## Let's define a problem: Breast Cancer Classification

The 'Breast cancer wisconsin (diagnostic) dataset' is a toy dataset provided by scikit-learn with 569 samples and 30 features. The goal is to classify the samples into two classes: benign (0) and malignant (1).

<font color='red'>**Disclaimer**</font>: This dataset is a toy dataset for educational purposes. It is important in real world applications to analyze your data and understand potential biases and limitations of your dataset.

In [8]:
# import some super basic data
from sklearn.datasets import load_breast_cancer

# load the iris dataset
data = load_breast_cancer(as_frame=True)
print(data.keys())

dict_keys(['data', 'target', 'frame', 'target_names', 'DESCR', 'feature_names', 'filename', 'data_module'])


In [15]:
print(type(data.data))
data.data

<class 'pandas.core.frame.DataFrame'>


,mean radius,mean texture,mean perimeter,mean area,mean smoothness,mean compactness,mean concavity,mean concave points,mean symmetry,mean fractal dimension,...,worst radius,worst texture,worst perimeter,worst area,worst smoothness,worst compactness,worst concavity,worst concave points,worst symmetry,worst fractal dimension
0,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.30010,0.14710,0.2419,0.07871,...,25.380,17.33,184.60,2019.0,0.16220,0.66560,0.7119,0.2654,0.4601,0.11890
1,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.08690,0.07017,0.1812,0.05667,...,24.990,23.41,158.80,1956.0,0.12380,0.18660,0.2416,0.1860,0.2750,0.08902
2,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.19740,0.12790,0.2069,0.05999,...,23.570,25.53,152.50,1709.0,0.14440,0.42450,0.4504,0.2430,0.3613,0.08758
3,11.42,20.38,77.58,386.1,0.14250,0.28390,0.24140,0.10520,0.2597,0.09744,...,14.910,26.50,98.87,567.7,0.20980,0.86630,0.6869,0.2575,0.6638,0.17300
4,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.19800,0.10430,0.1809,0.05883,...,22.540,16.67,152.20,1575.0,0.13740,0.20500,0.4000,0.1625,0.2364,0.07678
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
564,21.56,22.39,142.00,1479.0,0.11100,0.11590,0.24390,0.13890,0.1726,0.05623,...,25.450,26.40,166.10,2027.0,0.14100,0.21130,0.4107,0.2216,0.2060,0.07115
565,20.13,28.25,131.20,1261.0,0.09780,0.10340,0.14400,0.09791,0.1752,0.05533,...,23.690,38.25,155.00,1731.0,0.11660,0.19220,0.3215,0.1628,0.2572,0.06637
566,16.60,28.08,108.30,858.1,0.08455,0.10230,0.09251,0.05302,0.1590,0.05648,...,18.980,34.12,126.70,1124.0,0.11390,0.30940,0.3403,0.1418,0.2218,0.07820
567,20.60,29.33,140.10,1265.0,0.11780,0.27700,0.35140,0.15200,0.2397,0.07016,...,25.740,39.42,184.60,1821.0,0.16500,0.86810,0.9387,0.2650,0.4087,0.12400


In [16]:
print(type(data.target))
print(data.target)

<class 'pandas.core.series.Series'>
0      0
1      0
2      0
3      0
4      0
      ..
564    0
565    0
566    0
567    0
568    1
Name: target, Length: 569, dtype: int64


In order to use this data effortlessly, we should have a dataset class that hands the data over to the neural network.

In [19]:
# create a Dataset class for the iris data

# import torch and the Dataset class
import torch
from torch.utils.data import Dataset

class BreastCancerDataset(Dataset):
    """This is a child of Dataset providing the data to the dataloader
    The init and getitem constructors are absolutely necessary"""
    def __init__(self, data, targets):
        super().__init__()
        self.data = torch.Tensor(data.values)
        self.targets = torch.Tensor(targets.values)
    
    def __len__(self):
        return self.data.shape[0]
    
    def __getitem__(self, idx):
        # for a given index, return the data and target
        return self.data[idx,:], self.targets[idx]

dataset = BreastCancerDataset(data.data, data.target)
print(dataset)

Now we have provided a way for pytorch to understand our data. We can now use the `DataLoader` to provide us with random batches of our data during training.

In [21]:
# now we can create a dataloader
from torch.utils.data import DataLoader
dataloader = DataLoader(dataset, batch_size=1, shuffle=True)

# now we can iterate over the dataloader
for data, target in dataloader:
    print(data)
    print(target)
    break

tensor([[8.5970e+00, 1.8600e+01, 5.4090e+01, 2.2120e+02, 1.0740e-01, 5.8470e-02,
         0.0000e+00, 0.0000e+00, 2.1630e-01, 7.3590e-02, 3.3680e-01, 2.7770e+00,
         2.2220e+00, 1.7810e+01, 2.0750e-02, 1.4030e-02, 0.0000e+00, 0.0000e+00,
         6.1460e-02, 6.8200e-03, 8.9520e+00, 2.2440e+01, 5.6650e+01, 2.4010e+02,
         1.3470e-01, 7.7670e-02, 0.0000e+00, 0.0000e+00, 3.1420e-01, 8.1160e-02]])
tensor([1.])


## What is a Tensor?
Tensors are THE data structure in machine learning. A tensor is a structure very similar to a numpy array, but it can be used on GPUs. And GPUs we really need if we don't want to get old waiting ...

<img src="https://visagetechnologies.com/app/uploads/2022/08/Tensors.png" width="400">

(source: https://visagetechnologies.com/tensors-in-computer-vision/)

A tensor has 3 attributes:
- shape
- data type
- device

these words you will often see in error messages, because PyTorch complains about shapes not matching, and data types and devices not being the same.

In [33]:
import numpy as np

example_array = np.array([1, 2, 3, 4, 5])
print("array: ", example_array)
print("shape: ", example_array.shape)
print("dtype: ", example_array.dtype)

array:  [1 2 3 4 5]
shape:  (5,)
dtype:  int64


Let's make a tensor from this array.

In [34]:
example_tensor = torch.tensor(example_array)

Many things will be similar to numpy.

In [35]:
print("tensor shape: ", example_tensor.shape)
print("tensor dtype: ", example_tensor.dtype)

tensor shape:  torch.Size([5])
tensor dtype:  torch.int64


What is new is the device. The device is the location of the tensor on the computer. It will either be the CPU or GPU. The GPU makes things faster, so we love it. There are currently 3 options:
- cpu
- cuda (for NVIDIA GPUs)
- mps (for some MAC GPUs)

In [36]:
print("tensor device: ", example_tensor.device)

tensor device:  cpu


## Ready to build a model?

Now that we have some data and an idea of what Tensors are, we can look at how to build a simple neural network in PyTorch. We will use the `torch.nn` module, which provides a simple way to create a neural network.

<img src="https://media.geeksforgeeks.org/wp-content/cdn-uploads/20230602113310/Neural-Networks-Architecture.png" alt="NN architecture" width="400"/>

(source: https://www.geeksforgeeks.org/artificial-neural-networks-and-its-applications/)

Some generally important characteristics of simple NNs are the input dimension, number of hidden layers, the number of hidden units, and the output dimension.

In [22]:
# input and output are determined by the dataset
n_input = dataset.data.shape[1] # 30 predictive features of the data
n_output = 1 # binary classification

# we can choose the number of hidden layers and the number of neurons in each layer
#n_layers = 2 # number of hidden layers
n_hidden = 8 # number of units in each layer
# remember that we have a small dataset, so we should avoid having a too big model

In [23]:
# let's build a little classifier

import torch.nn as nn
import torch.nn.functional as F

# define the network
class Classifier(nn.Module):
    def __init__(self, in_features:int, hidden_features:int, out_features:int):
        super().__init__()
        self.fc1 = nn.Linear( # define the first fully connected layer
            in_features,
            hidden_features
        )
        self.fc2 = nn.Linear( # define the second fully connected layer
            hidden_features,
            out_features
        )
        
    def forward(self, x):
        z = self.fc1(x) # apply the first fully connected layer
        z = F.relu(z) # apply the relu activation function (non-linearity)
        z = self.fc2(z) # apply the second fully connected layer
        return F.sigmoid(z) # apply the sigmoid activation function for binary classification

In [24]:
bc_classifier = Classifier(in_features=n_input, hidden_features=n_hidden, out_features=n_output) # create an instance of the network
print(bc_classifier)

Classifier(
  (fc1): Linear(in_features=30, out_features=8, bias=True)
  (fc2): Linear(in_features=8, out_features=1, bias=True)
)


Now we can give the model some data and get a prediction.

In [29]:
# now we can iterate over the dataloader
for data, target in dataloader:
    y = bc_classifier(data)
    print("target: ", target, "prediction: ", y)
    break

target:  tensor([1.]) prediction:  tensor([[3.2499e-10]], grad_fn=<SigmoidBackward0>)


Everything happening in this forward pass is just a bunch of matrix multiplications of Tensors and non-linear functions. Each linear layer has a weight matrix and a bias vector. These are parameterized Tensors, and will receive gradients during the backward pass.

In [38]:
bc_classifier.fc2.weight

Parameter containing:
tensor([[ 0.0948, -0.2480, -0.2677, -0.3433,  0.2338, -0.0243, -0.0321, -0.2138]],
       requires_grad=True)